Here's the master notebooks for this project, connecting a bunch of separate python scripts.
If this project was to be continued to run as a scheduled job, I'd have built this as a script as well.

In [1]:
#We have a BUNCH of modules, so let's make sure we reload them automatically when we edit them.
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../src")

Time for imports. This is a bit leaner that would be expected due to the modular project of this project.
We also build in some toggles, since this is the master interface.

When running this for the first time, set RESET_DB and SCRAPE to True.

In order to change the list of cities, just edit the list below.

In [ ]:
import pandas as pd
from db import get_engine, insert_ignore
from schema import create_tables, reset_database
from scrape import cityscraper, popscraper
from weather import load_cities, get_weather_data, weather_to_df, get_weather_forecast, forecast_to_df

RESET_DB = False     # True wipes everything
SCRAPE   = False    # True re-scrapes Wikipedia (only needed for new cities)
euro_cities = ["Berlin", "Hamburg", "Munich", "Cologne", "Frankfurt", "London",
               "Kiruna", "Stockholm", "Prague", "Paris", "Kiev", "Madrid"]

In [3]:
#And here is where we execute the reset. It is still locked behind the safety above.

if RESET_DB:
    reset_database()
create_tables()
engine = get_engine()

In [4]:
#Here's the cell for running the webscraping. It is also locked behind the safety.
if SCRAPE:
    cities_df = cityscraper(euro_cities) #scrapes city country and coords from wikipedia into a dataframe
    pops_df = popscraper(euro_cities) #scrapes city population from wikipedia into a dataframe

    cities_df[["city_name"]].to_sql("cities", con=engine, if_exists="append",
                                    index=False, method=insert_ignore)
    keys = pd.read_sql("cities", con=engine)
    #the above injects the results into the database, and then reads back the city_id primary keys for the next step.

    cities_df.merge(keys, on="city_name")[["city_id", "longitude", "latitude"]] \
        .to_sql("cities_coords", con=engine, if_exists="append", index=False, method=insert_ignore)
    pops_df.merge(keys, on="city_name")[["city_id", "population", "year_retrieved"]] \
        .to_sql("city_pop", con=engine, if_exists="append", index=False, method=insert_ignore)
    #the above populates two more tables, using the key we just pulled

In [5]:
#and the last part for now, pulling and injecting weather data with an api from openweathermap.org. This is not locked behind a safety, because it is safe to run multiple times.
cities = load_cities(engine)

weather_df = weather_to_df(get_weather_data(cities))
forecasts_df = forecast_to_df(get_weather_forecast(cities))

n1 = weather_df.to_sql("weather_actual", con=engine, if_exists="append", index=False, method=insert_ignore)
n2 = forecasts_df.to_sql("weather_forecast", con=engine, if_exists="append", index=False, method=insert_ignore)
print(f"weather_actual: {n1} new rows, weather_forecast: {n2} new rows")

weather_actual: 12 new rows, weather_forecast: 480 new rows


In [6]:
#lets test
pd.read_sql("""
    SELECT c.city_name, w.measured_at, w.temp, w.weather_main
    FROM weather_actual w JOIN cities c USING (city_id)
    ORDER BY w.measured_at DESC LIMIT 12
""", con=engine)

,city_name,measured_at,temp,weather_main
0,Stockholm,2026-09-23 13:55:45,16.63,Clouds
1,Prague,2026-09-23 13:55:44,17.68,Clouds
2,Paris,2026-09-23 13:55:43,26.53,Clear
3,Munich,2026-09-23 13:55:42,19.31,Clear
4,Kiruna,2026-09-23 13:55:40,11.97,Clouds
5,Frankfurt,2026-09-23 13:55:38,20.89,Clouds
6,Cologne,2026-09-23 13:55:37,20.55,Clouds
7,Hamburg,2026-09-23 13:55:10,17.90,Clouds
8,London,2026-09-23 13:51:59,23.25,Clouds
9,Kiev,2026-09-23 13:51:49,13.52,Clouds
